# EDA 04 — Revenus des Menages (INSEE Filosofi IRIS 2021)
**Source** : INSEE Filosofi | **Bronze** : `data/bronze/revenus/`

In [ ]:

import os, glob, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
warnings.filterwarnings("ignore")

PALETTE = {
    "primary":   "#1565C0",
    "secondary": "#E53935",
    "accent":    "#2E7D32",
    "neutral":   "#546E7A",
}

plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.framealpha": 0.9,
    "legend.fontsize": 10,
})

BRONZE_ROOT = os.path.abspath("../../data/bronze")
NB_DIR      = os.path.abspath(".")
FIGURES_DIR = os.path.join(NB_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def save_fig(name, source_prefix, tight=True):
    dest = os.path.join(FIGURES_DIR, source_prefix)
    os.makedirs(dest, exist_ok=True)
    path = os.path.join(dest, f"{name}.png")
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  Saved -> figures/{source_prefix}/{name}.png")

print("Setup OK — figures ->", FIGURES_DIR)


In [ ]:

def draw_schema(title, columns, source_prefix, filename="schema"):
    n_rows = len(columns)
    fig_h  = max(3.0, 0.48 * n_rows + 1.6)
    fig, ax = plt.subplots(figsize=(13, fig_h))
    ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold", y=0.99, ha="center")
    headers = ["Colonne", "Type", "Description"]
    col_x   = [0.0, 0.22, 0.37]
    col_w   = [0.22, 0.15, 0.63]
    row_h   = 1.0 / (n_rows + 1)
    for i, (hdr, x, w) in enumerate(zip(headers, col_x, col_w)):
        rect = mpatches.FancyBboxPatch(
            (x, 1 - row_h), w - 0.006, row_h * 0.88,
            boxstyle="round,pad=0.01", linewidth=0.5,
            edgecolor="#90A4AE", facecolor="#1565C0",
            transform=ax.transAxes, clip_on=False)
        ax.add_patch(rect)
        ax.text(x + w/2, 1 - row_h/2, hdr,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=11, fontweight="bold", color="white")
    type_colors = {
        "str":"#1565C0","int":"#2E7D32","float":"#6A1B9A",
        "datetime":"#E65100","bool":"#AD1457","date":"#00695C",
    }
    for ridx, (col_name, col_type, col_desc) in enumerate(columns):
        y_top = 1 - (ridx + 2) * row_h
        bg = "#F5F7FA" if ridx % 2 == 0 else "white"
        for x, w in zip(col_x, col_w):
            rect = mpatches.FancyBboxPatch(
                (x, y_top), w - 0.006, row_h * 0.88,
                boxstyle="round,pad=0.01", linewidth=0.5,
                edgecolor="#CFD8DC", facecolor=bg,
                transform=ax.transAxes, clip_on=False)
            ax.add_patch(rect)
        y_mid = y_top + row_h * 0.44
        tc = type_colors.get(col_type.split("[")[0].lower(), "#37474F")
        ax.text(col_x[0]+col_w[0]/2, y_mid, col_name,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=10, fontweight="bold", color="#1A237E")
        ax.text(col_x[1]+col_w[1]/2, y_mid, col_type,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=9, fontfamily="monospace", color=tc)
        ax.text(col_x[2]+0.01, y_mid, col_desc,
                transform=ax.transAxes, ha="left", va="center",
                fontsize=9.5, color="#37474F")
    plt.subplots_adjust(top=0.92, bottom=0)
    save_fig(filename, source_prefix)
    plt.show()


In [ ]:
PREFIX = "04_revenus"
draw_schema(
    "Bronze Schema — Revenus INSEE Filosofi (IRIS 2021)",
    [
        ("iris_code",        "str",      "Code IRIS (9 caracteres INSEE)"),
        ("commune_code",     "str",      "Code INSEE commune (751xx)"),
        ("arrondissement",   "int",      "Numero arrondissement (1-20)"),
        ("median_income",    "float",    "Revenu median annuel du menage (euros)"),
        ("gini_coefficient", "float",    "Coefficient de Gini — inegalites intra-IRIS (0-1)"),
        ("poverty_rate",     "float",    "Taux de pauvrete (% menages sous le seuil a 60%)"),
        ("year",             "int",      "Millesime de reference (2021)"),
        ("latitude",         "float",    "Latitude du centroide IRIS"),
        ("longitude",        "float",    "Longitude du centroide IRIS"),
        ("ingested_at",      "datetime", "Horodatage UTC d'ingestion"),
    ], PREFIX
)

In [ ]:
rev_files = sorted(glob.glob(f"{BRONZE_ROOT}/revenus/**/*.parquet", recursive=True))
if rev_files:
    df = pd.concat([pd.read_parquet(f) for f in rev_files], ignore_index=True)
else:
    rng = np.random.default_rng(42); n_iris = 300
    arr = np.repeat(np.arange(1,21),15)[:n_iris]
    base = {a:28000+(21-a)*1400+rng.normal(0,2500) for a in range(1,21)}
    df = pd.DataFrame({"iris_code":[f"751{a:02d}{i:04d}" for a,i in zip(arr,range(n_iris))],
        "commune_code":[f"751{a:02d}" for a in arr],"arrondissement":arr,
        "median_income":[max(14000,base[a]+rng.normal(0,3500)) for a in arr],
        "gini_coefficient":rng.uniform(0.25,0.50,n_iris),
        "poverty_rate":rng.uniform(4,35,n_iris),"year":2021,
        "latitude":48.8566+rng.uniform(-0.07,0.07,n_iris),"longitude":2.3522+rng.uniform(-0.08,0.08,n_iris)})
print(f"Shape : {df.shape}")
df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(15,5))
axes[0].hist(df["median_income"]/1000,bins=40,color=PALETTE["accent"],edgecolor="white",alpha=0.85)
axes[0].axvline(df["median_income"].median()/1000,color=PALETTE["secondary"],linewidth=2,
                label=f"Mediane : {df['median_income'].median()/1000:.1f} k euros")
axes[0].set_title("Distribution des revenus medians (IRIS)"); axes[0].set_xlabel("k euros/an"); axes[0].legend()
sc = axes[1].scatter(df["gini_coefficient"],df["poverty_rate"],
                     c=df["median_income"]/1000,cmap="RdYlGn",alpha=0.65,s=35,edgecolors="none")
axes[1].set_xlabel("Coefficient de Gini"); axes[1].set_ylabel("Taux de pauvrete (%)")
axes[1].set_title("Gini vs Taux de pauvrete (couleur = revenu median)")
plt.colorbar(sc,ax=axes[1],label="k euros/an")
save_fig("distribution_revenus", PREFIX); plt.show()

In [ ]:
arr_income = df.groupby("arrondissement").agg(revenu_median=("median_income","median"),pauvrete_moy=("poverty_rate","mean"))
fig, axes = plt.subplots(1,2,figsize=(16,6))
norm = (arr_income["revenu_median"]-arr_income["revenu_median"].min())/(arr_income["revenu_median"].max()-arr_income["revenu_median"].min())
axes[0].bar(arr_income.index.astype(str),arr_income["revenu_median"]/1000,color=plt.cm.RdYlGn(norm),edgecolor="white")
axes[0].set_xlabel("Arrondissement"); axes[0].set_ylabel("k euros/an"); axes[0].set_title("Revenu median par arrondissement")
plt.setp(axes[0].xaxis.get_majorticklabels(),rotation=45)
norm2 = (arr_income["pauvrete_moy"]-arr_income["pauvrete_moy"].min())/(arr_income["pauvrete_moy"].max()-arr_income["pauvrete_moy"].min())
axes[1].bar(arr_income.index.astype(str),arr_income["pauvrete_moy"],color=plt.cm.Reds(0.3+norm2*0.6),edgecolor="white")
axes[1].set_xlabel("Arrondissement"); axes[1].set_ylabel("Taux de pauvrete (%)"); axes[1].set_title("Taux de pauvrete moyen par arrondissement")
plt.setp(axes[1].xaxis.get_majorticklabels(),rotation=45)
save_fig("revenus_par_arrondissement", PREFIX); plt.show()